In [1]:
import pandas as pd

df = pd.read_parquet("data/reactions.parquet")
df.head()

,id,timestamp,model_a_name,model_b_name,refers_to_model,msg_index,opening_msg,conversation_a,conversation_b,model_pos,...,creative,complete,clear_formatting,incorrect,superficial,instructions_not_followed,model_pair_name,msg_rank,question_id,system_prompt
0,202099,2025-04-23 19:32:46.687338,gemma-3-12b,command-a,command-a,1,Quelle est la stratégie commerciale du champag...,[{'content': 'Quelle est la stratégie commerci...,[{'content': 'Quelle est la stratégie commerci...,b,...,False,True,False,False,False,False,"[command-a, gemma-3-12b]",0,1a62544c50474085b25240991e9fc620-6e6ffca415f44...,NaN
1,176467,2025-03-25 13:41:21.988182,llama-3.1-8b,claude-3-5-sonnet-v2,llama-3.1-8b,1,Qui est président en France ?,"[{'content': 'Qui est président en France ?', ...","[{'content': 'Qui est président en France ?', ...",a,...,False,True,False,False,False,False,"[claude-3-5-sonnet-v2, llama-3.1-8b]",0,ba089364a6e64d14b80eae6f7dd1885f-cda152aa7fe04...,NaN
2,231166,2025-06-03 15:38:11.580715,gpt-4.1-nano,llama-4-scout,llama-4-scout,1,tests logiciel,"[{'content': 'tests logiciel', 'metadata': Non...","[{'content': 'tests logiciel', 'metadata': Non...",b,...,False,True,False,False,False,False,"[gpt-4.1-nano, llama-4-scout]",0,e46e59300e8545038bfc22dfffb0b746-b006443eb1564...,NaN
3,264589,2025-08-20 13:52:02.676745,llama-3.1-405b,mistral-large-2411,llama-3.1-405b,13,Vols au meilleur prix au départ de Marseille p...,[{'content': 'Vols au meilleur prix au départ ...,[{'content': 'Vols au meilleur prix au départ ...,a,...,False,False,False,False,False,False,"[llama-3.1-405b, mistral-large-2411]",6,1f181e340b664f88b1aacc6770d0eeff-3f295419957c4...,NaN
4,203152,2025-04-25 20:40:31.489534,o4-mini,claude-3-7-sonnet,o4-mini,1,Comporte toi comme un expert en rédaction de p...,[{'content': 'Comporte toi comme un expert en ...,[{'content': 'Comporte toi comme un expert en ...,a,...,False,False,False,False,True,False,"[claude-3-7-sonnet, o4-mini]",0,5a646fd23c7c480dbbb9dac5f50b8c6d-5d343345ef294...,NaN


In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 91833 entries, 0 to 91832
Data columns (total 33 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   id                         91833 non-null  int64         
 1   timestamp                  91833 non-null  datetime64[us]
 2   model_a_name               91833 non-null  str           
 3   model_b_name               91833 non-null  str           
 4   refers_to_model            91833 non-null  str           
 5   msg_index                  91833 non-null  int64         
 6   opening_msg                91833 non-null  str           
 7   conversation_a             91833 non-null  object        
 8   conversation_b             91833 non-null  object        
 9   model_pos                  91833 non-null  str           
 10  conv_turns                 91833 non-null  int64         
 11  conversation_pair_id       91833 non-null  str           
 12  conv_a_id      

In [3]:
grouped = df.groupby(["conversation_pair_id", "msg_index"])\
    .filter(lambda x: len(x) > 1)\
    .groupby(["conversation_pair_id", "msg_index"])
# Keep only indexes columns and creative
grouped = grouped[["conversation_pair_id", "msg_index", "creative", "visitor_id"]]
grouped.head()

,conversation_pair_id,msg_index,creative,visitor_id
1,ba089364a6e64d14b80eae6f7dd1885f-cda152aa7fe04...,1,False,3d3720ea8c5ff9dfbdbc886bd7ee7d00
3,1f181e340b664f88b1aacc6770d0eeff-3f295419957c4...,13,False,1299d615674a3add049c9869cf9eac1b
4,5a646fd23c7c480dbbb9dac5f50b8c6d-5d343345ef294...,1,False,36ffd7b8de35fe20f4850fa9f45d88e1
5,ba089364a6e64d14b80eae6f7dd1885f-cda152aa7fe04...,1,False,3d3720ea8c5ff9dfbdbc886bd7ee7d00
7,5a646fd23c7c480dbbb9dac5f50b8c6d-5d343345ef294...,1,False,36ffd7b8de35fe20f4850fa9f45d88e1
...,...,...,...,...
91828,870599a3a9be4ee689757ccfb761a161-845bc8f6b3c34...,1,False,fc56866e083476414e83d6f9e3c7f278
91829,f90b121865a74e0e9250dc598597f108-2531f6d6f3bf4...,1,False,fc56866e083476414e83d6f9e3c7f278
91830,f90b121865a74e0e9250dc598597f108-2531f6d6f3bf4...,1,False,fc56866e083476414e83d6f9e3c7f278
91831,89eced1c56ae48709cfc8459881e4a12-0bb85709c7714...,1,False,fc56866e083476414e83d6f9e3c7f278


In [22]:
# For each group, check if all creatives are the same
results = []
def check_creatives(group):
    creatives = group["creative"].unique()
    results.append({
        "conversation_pair_id": group["conversation_pair_id"].iloc[0],
        "msg_index": group["msg_index"].iloc[0],
        "all_same": len(creatives) == 1,
        "creatives": creatives,
        "nb_reactions": len(group)
    })
grouped.apply(check_creatives)
results_df = pd.DataFrame(results)
results_df.head()

,conversation_pair_id,msg_index,all_same,creatives,nb_reactions
0,00034204d6994d8e99070b75b4fb3f67-2e5d7407743f4...,1,False,"[False, True]",2
1,00072aba380842e1afea00f70b01d9e7-fc968fd0b7f34...,1,True,[False],2
2,00084ec9d4b54b20841ab7decd9240be-3db325799f764...,1,True,[False],2
3,000c9b3eb2bd41479d606579dcffa137-1a8f8af84d734...,1,True,[False],2
4,000ed32e609345a6ba595acacc23678f-58424a6cb5e64...,1,True,[False],2


In [26]:
print(f"Number of common messages: {len(results_df)}")
print(f"Ratio of common messages with all creatives the same: {results_df['all_same'].mean():.2%}")

Number of common messages: 31868
Ratio of common messages with all creatives the same: 90.61%
